# Reinforcement Learning 소개 · 학습


## 실행 조건

<!-- exercise-guidance:reinforcement-learning/week-01/execution -->
기준 Queue 예제는 NumPy로 실행합니다. 학습에 사용하지 않은 seed=100…104에서 정책을 비교합니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/execution -->


## 이번에 배울 내용

<!-- exercise-guidance:reinforcement-learning/week-01/learning-intro -->
보상을 잘 받는 정책이 실제로 원하는 운전인지, 비용과 대기를 분리해 확인합니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/learning-intro -->


## 보상과 실제 목적

<!-- exercise-guidance:reinforcement-learning/week-01/learning-1 -->
Return은 보상 설계의 결과입니다. 원래 단위의 대기·비용·유실을 분리해 봐야 높은 Return이 실제 목적에 맞는지 판단할 수 있습니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/learning-1 -->


## 관측과 상태

<!-- exercise-guidance:reinforcement-learning/week-01/learning-2 -->
현재 Queue 길이와 도착률처럼 의사결정 시점에 얻을 수 있는 값을 Observation으로 사용합니다. 미래의 실제 도착 건수는 Agent가 아직 알 수 없는 값입니다. 같은 Observation과 Action에서 다음 상태의 확률분포를 정할 수 있는지 살펴봅니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/learning-2 -->


## 풀이 예제

<!-- exercise-guidance:reinforcement-learning/week-01/learning-example -->
Queue의 두 정책을 기준으로 비용 계수 0.2의 의미를 살펴봅니다.
<!-- /exercise-guidance:reinforcement-learning/week-01/learning-example -->


In [1]:
"""Synthetic service queue. An action chooses capacity, including its energy cost."""
import numpy as np

class Queue:
    def reset(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.q, self.t = 0, 0
        return self.q

    def step(self, action):
        capacity = (0, 2, 4)[int(action)]
        arriving = int(self.rng.poisson(2))
        raw = max(0, self.q - capacity) + arriving
        dropped = max(0, raw - 20)
        self.q = min(20, raw)
        self.t += 1
        reward = -(self.q + .2 * capacity**2 + 10 * dropped)
        return self.q, reward, False, self.t >= 100, {'queue': self.q, 'capacity': capacity, 'dropped': dropped}

def evaluate(policy, seeds=range(100, 105)):
    rows = []
    for seed in seeds:
        env = Queue(); q = env.reset(seed)
        total = waiting = energy = dropped = 0
        while True:
            q, r, terminated, truncated, info = env.step(policy(q))
            total += r; waiting += q; energy += info['capacity']**2; dropped += info['dropped']
            if terminated or truncated: break
        rows.append([total, waiting / env.t, energy / env.t, dropped])
    return np.array(rows)

policies = {'constant': lambda q: 1, 'threshold': lambda q: 0 if q == 0 else (1 if q < 5 else 2)}
for name, policy in policies.items():
    print(name, evaluate(policy).mean(axis=0))


constant [-921.2      8.052    4.       3.6  ]
threshold [-387.4     2.77    5.52    0.  ]


## 참고 예제

<!-- exercise-guidance:reinforcement-learning/week-01/learning-references -->
<details class="learning-references"><summary>추가 예제 보기</summary><ul><li><a data-course-reference="" href="https://chorok-daddy.github.io/niil/courses/reinforcement-learning/week-01/RL_W1.preview.html">Reinforcement Learning 소개</a></li></ul></details>
<!-- /exercise-guidance:reinforcement-learning/week-01/learning-references -->
